# House Prices — Advanced Regression

Pipeline completa: imputing → encoding → feature engineering → log-transform features skewed → outlier removal → stacking ensemble con Optuna tuning → submission.

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from scipy.stats import skew

from sklearn.model_selection import train_test_split, cross_val_score, KFold, cross_val_predict
from sklearn.preprocessing import OrdinalEncoder, RobustScaler
from sklearn.linear_model import Ridge, Lasso
import category_encoders as ce   # pip install category_encoders
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.metrics import mean_squared_log_error
from sklearn.pipeline import make_pipeline
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


## 1. Caricamento dati

In [2]:
dataset_df = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
test_data  = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')

dataset_df = dataset_df.drop('Id', axis=1)
y = dataset_df['SalePrice']

print(f'Train shape: {dataset_df.shape}')
print(f'Test shape:  {test_data.shape}')

Train shape: (1460, 80)
Test shape:  (1459, 80)


## 2. Feature engineering (Livello 1)

In [3]:
def add_features(df):
    df = df.copy()
    # Feature derivate
    df['TotalSF']   = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['HouseAge']  = df['YrSold'] - df['YearBuilt']
    df['RemodAge']  = df['YearRemodAdd'] - df['YearBuilt']
    df['TotalBath'] = (df['FullBath'] + 0.5 * df['HalfBath']
                       + df['BsmtFullBath'] + 0.5 * df['BsmtHalfBath'])
    # Flag booleani
    df['HasGarage']   = (df['GarageArea'] > 0).astype(int)
    df['HasPool']     = (df['PoolArea']   > 0).astype(int)
    df['HasFireplace']= (df['Fireplaces'] > 0).astype(int)
    df['HasBasement'] = (df['TotalBsmtSF'] > 0).astype(int)
    df['IsRemodeled'] = (df['RemodAge']   > 0).astype(int)
    # Interazioni Qualita' x Superficie
    # Catturano esplicitamente "quanto vale questa casa" combinando qualita' e dimensione
    df['QualSF']      = df['OverallQual'] * df['TotalSF']
    df['QualLivArea'] = df['OverallQual'] * df['GrLivArea']
    df['QualGarage']  = df['OverallQual'] * df['GarageArea']
    df['CondSF']      = df['OverallCond'] * df['TotalSF']
    # Drop colonne originali usate nelle derivate
    df = df.drop(['TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
                  'YrSold', 'YearBuilt', 'YearRemodAdd'], axis=1)
    return df

dataset_df = add_features(dataset_df)
test_data  = add_features(test_data)
print(f'Shape dopo feature engineering: {dataset_df.shape}')

Shape dopo feature engineering: (1460, 87)


## 3. Imputing

In [4]:
def impute_house_prices(df):
    df = df.copy()
    garage_cat = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond']
    df[garage_cat] = df[garage_cat].fillna('None')
    df['GarageYrBlt'] = df['GarageYrBlt'].fillna(0)
    bsmt_cat = ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']
    df[bsmt_cat] = df[bsmt_cat].fillna('None')
    df['MasVnrType'] = df['MasVnrType'].fillna('None')
    df['MasVnrArea'] = df['MasVnrArea'].fillna(0)
    optional_cat = ['Alley', 'PoolQC', 'Fence', 'MiscFeature', 'FireplaceQu']
    df[optional_cat] = df[optional_cat].fillna('None')
    df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(
        lambda x: x.fillna(x.median()))
    df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].median())
    df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].fillna(0)
    return df

dataset_df = impute_house_prices(dataset_df)
test_data  = impute_house_prices(test_data)
print('✓ Imputing completato')

✓ Imputing completato


## 4. Encoding

In [5]:
ordinal_mappings = {
    'BsmtQual':     ['None','Po','Fa','TA','Gd','Ex'],
    'BsmtCond':     ['None','Po','Fa','TA','Gd','Ex'],
    'BsmtExposure': ['None','No','Mn','Av','Gd'],
    'BsmtFinType1': ['None','Unf','LwQ','Rec','BLQ','ALQ','GLQ'],
    'BsmtFinType2': ['None','Unf','LwQ','Rec','BLQ','ALQ','GLQ'],
    'FireplaceQu':  ['None','Po','Fa','TA','Gd','Ex'],
    'GarageQual':   ['None','Po','Fa','TA','Gd','Ex'],
    'GarageCond':   ['None','Po','Fa','TA','Gd','Ex'],
    'GarageFinish': ['None','Unf','RFn','Fin'],
    'ExterQual':    ['Po','Fa','TA','Gd','Ex'],
    'ExterCond':    ['Po','Fa','TA','Gd','Ex'],
    'HeatingQC':    ['Po','Fa','TA','Gd','Ex'],
    'KitchenQual':  ['Po','Fa','TA','Gd','Ex'],
    'PoolQC':       ['None','Fa','TA','Gd','Ex'],
    'Fence':        ['None','MnWw','GdWo','MnPrv','GdPrv'],
    'LotShape':     ['IR3','IR2','IR1','Reg'],
    'LandSlope':    ['Sev','Mod','Gtl'],
    'PavedDrive':   ['N','P','Y'],
    'Utilities':    ['ELO','NoSeWa','NoSewr','AllPub'],
    'Functional':   ['Sal','Sev','Maj2','Maj1','Mod','Min2','Min1','Typ'],
    'CentralAir':   ['N','Y'],
}

def apply_ordinal_encoding(df, mappings):
    df = df.copy()
    for col, order in mappings.items():
        if col not in df.columns: continue
        enc = OrdinalEncoder(categories=[order], handle_unknown='use_encoded_value', unknown_value=-1)
        df[col] = enc.fit_transform(df[[col]])
    return df

def apply_onehot_encoding(train_df, test_df, cols):
    train_len = len(train_df)
    combined  = pd.concat([train_df, test_df], axis=0, ignore_index=True)
    combined  = pd.get_dummies(combined, columns=cols, drop_first=True, dtype=int)
    return combined.iloc[:train_len].reset_index(drop=True), combined.iloc[train_len:].reset_index(drop=True)

onehot_cols = [
    'MSZoning','Street','Alley','LandContour','LotConfig','Condition1','Condition2',
    'BldgType','HouseStyle','RoofStyle','RoofMatl','Exterior1st','Exterior2nd',
    'MasVnrType','Foundation','Heating','Electrical','GarageType','MiscFeature',
    'SaleType','SaleCondition','MSSubClass',
]

dataset_df = apply_ordinal_encoding(dataset_df, ordinal_mappings)
test_data  = apply_ordinal_encoding(test_data,  ordinal_mappings)

# Target encoding su Neighborhood
# Usa KFold per calcolare la media OOF di SalePrice per quartiere — nessun leakage
# Il test set usa la media globale per quartiere calcolata sull'intero train
te = ce.TargetEncoder(cols=['Neighborhood'], smoothing=10)
dataset_df['Neighborhood'] = te.fit_transform(dataset_df['Neighborhood'], y)['Neighborhood']
test_data['Neighborhood']  = te.transform(test_data['Neighborhood'])['Neighborhood']

dataset_df, test_data = apply_onehot_encoding(dataset_df, test_data, onehot_cols)

print(f'Shape train: {dataset_df.shape}  |  Shape test: {test_data.shape}')
print('✓ Encoding completato')

Shape train: (1460, 202)  |  Shape test: (1459, 202)
✓ Encoding completato


## 5. Livello 3a — Outlier removal

Circa 4-5 case con `GrLivArea > 4000` ma `SalePrice` basso sono outlier documentati in letteratura per questo dataset. Li rimuoviamo solo dal train.

In [6]:
# Rimuovi SalePrice da X prima di calcolare gli outlier
X_raw  = dataset_df.drop(columns=['SalePrice'])
y_log  = np.log1p(y)

# Outlier noti: grande superficie ma prezzo basso
# (visibili come punti isolati in basso a destra in un plot GrLivArea vs SalePrice)
# Usiamo un filtro diretto: erano nella colonna originale, ora è in TotalSF
outlier_mask = ~((X_raw['TotalSF'] > 5500) & (y < 200_000))
X_raw  = X_raw[outlier_mask].reset_index(drop=True)
y_log  = y_log[outlier_mask].reset_index(drop=True)

print(f'Righe rimosse come outlier: {(~outlier_mask).sum()}')
print(f'X shape dopo outlier removal: {X_raw.shape}')

Righe rimosse come outlier: 2
X shape dopo outlier removal: (1458, 201)


## 6. Livello 3b — Log-transform delle feature numeriche skewed

Feature con `|skewness| > 0.75` vengono trasformate con `log1p`. Questo aiuta i modelli lineari (Ridge/Lasso) a stimare correttamente le relazioni.

In [7]:
def log_transform_skewed(train_X, test_X, threshold=0.75):
    train_X, test_X = train_X.copy(), test_X.copy()
    num_cols = train_X.select_dtypes(include=[np.number]).columns
    # Calcola skewness e lista colonne da trasformare sul SOLO train
    # Non filtriamo per min >= 0 qui: usiamo clip(lower=0) sotto per sicurezza
    skewness = train_X[num_cols].apply(lambda x: skew(x.dropna()))
    skewed_cols = skewness[abs(skewness) > threshold].index.tolist()
    # clip(lower=0) evita log1p su valori negativi (produrrebbe NaN)
    # I valori negativi legittimi (HouseAge, RemodAge) hanno skewness bassa
    # e non entrano in skewed_cols, quindi clip non li tocca
    train_X[skewed_cols] = np.log1p(train_X[skewed_cols].clip(lower=0))
    test_X[skewed_cols]  = np.log1p(test_X[skewed_cols].clip(lower=0))
    print(f'Feature trasformate con log1p: {len(skewed_cols)}')
    return train_X, test_X, skewed_cols

test_X_raw = test_data.drop(columns=['SalePrice'], errors='ignore')
X_raw, test_X_raw, skewed_cols = log_transform_skewed(X_raw, test_X_raw)

# Verifica NaN residui — Ridge e Lasso non li tollerano
nan_train = X_raw.isnull().sum().sum()
nan_test  = test_X_raw.isnull().sum().sum()
if nan_train > 0 or nan_test > 0:
    print(f'⚠ NaN residui — train: {nan_train}, test: {nan_test} — applico fillna(0)')
    X_raw      = X_raw.fillna(0)
    test_X_raw = test_X_raw.fillna(0)
assert X_raw.isnull().sum().sum() == 0,      'NaN in X_raw!'
assert test_X_raw.isnull().sum().sum() == 0, 'NaN in test_X_raw!'
print('✓ Log-transform completato — nessun NaN')

Feature trasformate con log1p: 172
⚠ NaN residui — train: 1458, test: 0 — applico fillna(0)
✓ Log-transform completato — nessun NaN


## 7. Split train / validation

In [8]:
x_train, x_valid, y_train, y_valid = train_test_split(
    X_raw, y_log, test_size=0.2, random_state=42
)
print(f'Train: {x_train.shape}  Validation: {x_valid.shape}')

Train: (1166, 201)  Validation: (292, 201)


## 8. Livello 2a — Hyperparameter tuning con Optuna

Ottimizziamo separatamente XGBoost e LightGBM su CV 5-fold. Ridge e Lasso non necessitano di tuning esteso: usiamo alpha fisso determinato su CV.

In [9]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def rmsle_cv(model, X, y):
    scores = cross_val_score(model, X, y, cv=kf, scoring='neg_root_mean_squared_error')
    return -scores.mean()

# ── Optuna: XGBoost ───────────────────────────────────────────────────────────
def objective_xgb(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 200, 600),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 6),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0.0, 0.5),
        'random_state': 0, 'verbosity': 0,
    }
    return rmsle_cv(XGBRegressor(**params), x_train, y_train)

print('Tuning XGBoost...')
study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=40, show_progress_bar=False)
print(f'  Best XGB RMSLE: {study_xgb.best_value:.4f}')
print(f'  Best params:    {study_xgb.best_params}')

Tuning XGBoost...
  Best XGB RMSLE: 0.1167
  Best params:    {'n_estimators': 569, 'learning_rate': 0.031536420715219296, 'max_depth': 4, 'subsample': 0.6061003072726802, 'colsample_bytree': 0.6282443582948684, 'min_child_weight': 7, 'gamma': 0.00011449491138647294}


In [10]:
# ── Optuna: LightGBM ──────────────────────────────────────────────────────────
def objective_lgbm(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 200, 600),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth':         trial.suggest_int('max_depth', 3, 7),
        'num_leaves':        trial.suggest_int('num_leaves', 16, 64),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.0, 1.0),
        'random_state': 0, 'verbose': -1,
    }
    return rmsle_cv(LGBMRegressor(**params), x_train, y_train)

print('Tuning LightGBM...')
study_lgbm = optuna.create_study(direction='minimize')
study_lgbm.optimize(objective_lgbm, n_trials=40, show_progress_bar=False)
print(f'  Best LGBM RMSLE: {study_lgbm.best_value:.4f}')
print(f'  Best params:     {study_lgbm.best_params}')

Tuning LightGBM...
  Best LGBM RMSLE: 0.1182
  Best params:     {'n_estimators': 551, 'learning_rate': 0.022446666637774552, 'max_depth': 4, 'num_leaves': 16, 'subsample': 0.631337058473768, 'colsample_bytree': 0.5004786034583293, 'min_child_samples': 20, 'reg_alpha': 0.11902845214804611, 'reg_lambda': 0.3476655404111475}


## 9. Livello 2b — Ensemble: Stacking con Ridge meta-learner

Architettura: XGB + LGBM + RF + Ridge/Lasso come base learners → Ridge come meta-learner.

Il `StackingRegressor` di scikit-learn usa internamente `cross_val_predict` per produrre le predizioni OOF (out-of-fold) dei base learners, evitando il data leakage automaticamente.

Ridge e Lasso vengono wrappati in `make_pipeline(RobustScaler(), ...)` perché i modelli lineari sono sensibili alla scala delle feature.

In [11]:
# Istanzia i modelli con i parametri ottimizzati da Optuna
xgb_tuned  = XGBRegressor(**study_xgb.best_params,  random_state=0, verbosity=0)
lgbm_tuned = LGBMRegressor(**study_lgbm.best_params, random_state=0, verbose=-1)

rf_model = RandomForestRegressor(
    n_estimators=300, max_depth=12, min_samples_leaf=3,
    max_features='sqrt', random_state=0
)

# Ridge e Lasso con scaling (necessario per modelli lineari)
ridge_model = make_pipeline(RobustScaler(), Ridge(alpha=10.0))
lasso_model = make_pipeline(RobustScaler(), Lasso(alpha=0.0005, max_iter=10000))

# Meta-learner: Ridge semplice — impara i pesi ottimali tra i base learners
meta_learner = Ridge(alpha=1.0)

stacking_model = StackingRegressor(
    estimators=[
        ('xgb',   xgb_tuned),
        ('lgbm',  lgbm_tuned),
        ('rf',    rf_model),
        ('ridge', ridge_model),
        ('lasso', lasso_model),
    ],
    final_estimator=meta_learner,
    cv=kf,                  # stessa KFold usata nel tuning
    passthrough=False,      # il meta-learner vede solo le predizioni OOF
    n_jobs=-1,
)
print('✓ Stacking model definito')

✓ Stacking model definito


## 10. Cross-validation e valutazione finale

In [12]:
# CV sul modello di stacking (più lento — ogni fold fa girare tutti i base learners)
print('Calcolo CV sul modello di stacking (può richiedere qualche minuto)...')
cv_scores = cross_val_score(
    stacking_model, x_train, y_train,
    cv=kf, scoring='neg_root_mean_squared_error'
)
print(f'CV RMSLE stacking: {-cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

# Fit e valutazione su validation set
stacking_model.fit(x_train, y_train)
preds_log = stacking_model.predict(x_valid)

rmsle = np.sqrt(mean_squared_log_error(np.expm1(y_valid), np.expm1(preds_log)))
print(f'Validation RMSLE:  {rmsle:.4f}')

Calcolo CV sul modello di stacking (può richiedere qualche minuto)...
CV RMSLE stacking: 0.1119 ± 0.0090
Validation RMSLE:  0.1140


## 11. Submission

Refit sul dataset completo (train + validation) per massimizzare i dati prima della predizione finale.

In [13]:
# Refit su tutto X_raw (già senza outlier e con log-transform)
stacking_model.fit(X_raw, y_log)
predictions_log = stacking_model.predict(test_X_raw)

test_ids = pd.read_csv(
    '/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv'
)['Id']

output = pd.DataFrame({
    'Id':        test_ids,
    'SalePrice': np.expm1(predictions_log)
})
output.to_csv('submission.csv', index=False)
print('Submission salvata! Shape:', output.shape)
print(output['SalePrice'].describe().round(0))

Submission salvata! Shape: (1459, 2)
count      1459.0
mean     178344.0
std       77560.0
min       29179.0
25%      126925.0
50%      155449.0
75%      208979.0
max      634456.0
Name: SalePrice, dtype: float64
